In [ ]:
import sys
sys.path.append("../../reranchor")

In [ ]:
import fitz
from PIL import Image

def load_pdf(pdf_path, dpi=144):
    images = []
    doc = fitz.open(pdf_path)
    for i in range(len(doc)):
        page = doc[i]
        pix = page.get_pixmap(matrix=fitz.Matrix(dpi/72, dpi/72))
        image = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        if pix.width > 3000 or pix.height > 3000:
            pix = page.get_pixmap(matrix=fitz.Matrix(1, 1), alpha=False)
            image = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
            images.append(image)
    return images

In [ ]:
from tqdm import tqdm
from collections import defaultdict
import os
import json

dataset_dir = "/mnt/nas/ricky/reranchor_dataset/pdf-mvqa"

with open(os.path.join(dataset_dir, "formatted.json"), 'r') as f:
    ds = json.load(f)

pdf_page_image = defaultdict(list)
for item in tqdm(ds, total=len(ds)):
    pdf_id = item['pdf_id']
    if pdf_id in pdf_page_image:
        continue
    page = item['page']
    ori_pdf_path = f'/mnt/nas/stevekll/dataset/pdf_mvqa/pdf10000/{pdf_id}.pdf'
    page_images = load_pdf(ori_pdf_path)
    pdf_page_image[pdf_id] = page_images

In [ ]:
import json
with open(f'cache/retrieval_results.json', 'r') as f:
    retrieved_result = json.load(f)

In [ ]:
import torch
from colpali_engine.models import ColQwen2_5, ColQwen2_5_Processor
from reranchor_lib import Qwen2_5_RerAnchor, ReranchorProcessor
import os

os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"]="0,1,2,3"
embed_device = "cuda:0"
rerank_device = "cuda:0"

modelname = "vidore/colqwen2.5-v0.2"
model = ColQwen2_5.from_pretrained(
        modelname,
        device_map=embed_device,
    ).eval()
processor = ColQwen2_5_Processor.from_pretrained("vidore/colqwen2.5-v0.2")


rerank_modelname = "ricky42613/reranchor-qwen2.5-3b"
rerank_model = Qwen2_5_RerAnchor.from_pretrained(
    rerank_modelname,
    device_map=rerank_device, 
    torch_dtype=torch.bfloat16
)
rerank_model.eval()
rerank_processor = ReranchorProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct")

In [ ]:
from tqdm import tqdm
from reranchor_lib import denoise_screenshot
for i, item in tqdm(enumerate(retrieved_result), total=len(retrieved_result)):
    if item['ground_truth_doc'] not in item['retrieved_docs']:
        continue
    query = item['query']
    batch_queries = processor.process_queries([query]).to(model.device)
    with torch.no_grad():
        query_embed = model(**batch_queries)

    scores = []
    for doc in item['retrieved_docs']:
        screenshot_image = pdf_page_image[doc['pdf']][doc['page']]

        denoised_image = denoise_screenshot(rerank_processor, rerank_model, query, screenshot_image, k_tokens=200)
        batch_mask_images = processor.process_images([denoised_image]).to(model.device)
        mask_image_embed = model(**batch_mask_images)
        late_interaction_score = processor.score_multi_vector(query_embed, mask_image_embed)
        rerank_score = late_interaction_score[0][0]
        scores.append(rerank_score)
    retrieved_result[i]['rerank_docs'] = [doc for _, doc in sorted(zip(scores, item['retrieved_docs']), key=lambda x: x[0], reverse=True)]
    break

    

In [ ]:
import json

ret = {}
for k in [1, 3, 5, 10]:
    print(f"Evaluating Recall@{k} and NDCG@{k}")
    recall = 0
    mrr = 0
    for item in retrieved_result:
        gt = item['ground_truth_doc']
        rerank_docs = item['rerank_docs'][:k]
        if item['ground_truth_doc'] in rerank_docs:
            recall += 1
            gt_idx = rerank_docs.index(item['ground_truth_doc'])
            mrr += 1/(gt_idx+1)
            break

    recall = recall / len(retrieved_result)
    mrr = mrr / len(retrieved_result)
    ret[f"Recall@{k}"] = recall
    ret[f"MRR@{k}"] = mrr

with open(f"cache/reranchor_rerank_metrics.json", 'w') as f:
    json.dump(ret, f, indent=4)
